In [8]:
import subprocess
import tarfile

def run_unidock(
    receptor: str,
    ligand_index: str,
    center_x: float,
    center_y: float,
    center_z: float,
    size_x: float = 25,
    size_y: float = 25,
    size_z: float = 25,
    seed: int = 42,
    max_gpu_memory: int = 20000,
    num_modes: int = 1,
    verbosity: int = 1,
    cpu: int = 32,
    search_mode: str = "detail",
    scoring: str = "vina",
    output_dir: str = "output",
    log_file: str = "log.txt"
):
    cmd = [
        "unidock",
        "--receptor", receptor,
        "--ligand_index", ligand_index,
        "--center_x", str(center_x),
        "--center_y", str(center_y),
        "--center_z", str(center_z),
        "--size_x", str(size_x),
        "--size_y", str(size_y),
        "--size_z", str(size_z),
        "--seed", str(seed),
        "--max_gpu_memory", str(max_gpu_memory),
        "--num_modes", str(num_modes),
        "--verbosity", str(verbosity),
        "--cpu", str(cpu),
        "--search_mode", search_mode,
        "--scoring", scoring,
        "--dir", output_dir
    ]

    print(" ".join(cmd))

    with open(log_file, "w") as log:
        subprocess.run(cmd, stdout=log, stderr=subprocess.STDOUT)


In [9]:
import pandas as pd
import shutil
import os

# Define paths
root = '.'
# root = os.path.dirname(os.path.abspath(__file__))
UNIDOCK_PATH = os.path.join(root, "..", "processed", "unidock_docking")
OUTPATH = os.path.join(UNIDOCK_PATH, "docking_results")  # LOGS, SDFs, ?
os.makedirs(OUTPATH, exist_ok=True)

# Load pocket detection data
pocket_detection_data = pd.read_csv(os.path.join(root, "..", "processed", "pocket_detection_data.csv"))

# Shuffle with fixed seed
df = pocket_detection_data.sample(frac=1, random_state=42).reset_index(drop=True)

# # Untar conformations prepared
# shutil.unpack_archive(os.path.join(UNIDOCK_PATH, "conformations_prepared.tar.gz"), 
#                       os.path.join(UNIDOCK_PATH, "conformations_prepared"), 'gztar')


In [10]:
# For each pocket
for file, pocket_number, centroid in zip(df['File name'], df['Pocket number'], df['Pocket centroid coordinate (x y z)']):

    # Prints
    print(f"\n\n--- File name: {file}; Pocket number: {pocket_number} ---\n\n")

    # Create directories
    label = file.replace(".pdb", "") + "_pocket_" + str(pocket_number)
    outpath = os.path.join(OUTPATH, label)
    os.makedirs(os.path.join(outpath, "docking"), exist_ok=True)

    # Extract structure
    tarfile.open(os.path.join(UNIDOCK_PATH, "structures_prepared.tar.gz")).extract("./" + file.replace(".pdb", ".pdbqt"), 
                                                                                   path=outpath, filter='data')

    # Copy pocket SD file
    shutil.copyfile(os.path.join(root, "..", "processed", "pocketvec_PRE", label, f"pocket_{label.split('_')[-1]}.sd"), 
                    os.path.join(outpath, f"pocket_{label.split('_')[-1]}.sd"))
    

    # Prepare docking variables
    receptor = os.path.join(outpath, file.replace(".pdb", ".pdbqt"))
    center_x, center_y, center_z = centroid.split()
    ligand_index = os.path.join(UNIDOCK_PATH, "input_ligands.txt")
    search_mode = 'fast'
    output_dir = os.path.join(outpath, "docking")
    log_file = os.path.join(outpath, "logs.log")

    # Run docking
    run_unidock(receptor=receptor, ligand_index=ligand_index, center_x=center_x, center_y=center_y, center_z=center_z,
                search_mode=search_mode, output_dir=output_dir, log_file=log_file)




    break



--- File name: chai1_P9WFS9_model_3.pdb; Pocket number: 4 ---


unidock --receptor ./../processed/unidock_docking/docking_results/chai1_P9WFS9_model_3_pocket_4/chai1_P9WFS9_model_3.pdbqt --ligand_index ./../processed/unidock_docking/input_ligands.txt --center_x 4.7605 --center_y 21.6655 --center_z 24.485 --size_x 25 --size_y 25 --size_z 25 --seed 42 --max_gpu_memory 20000 --num_modes 1 --verbosity 1 --cpu 32 --search_mode fast --scoring vina --dir ./../processed/unidock_docking/docking_results/chai1_P9WFS9_model_3_pocket_4/docking
